In [247]:
import fine as fn
from getData import getData
from pathlib import Path
import pandas as pd
import numpy as np


cwd = Path.cwd()
data = getData()

%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [248]:
locations = {"Region1", "Region2", "Region3"}
commodityUnitDict = {"electricity": r"GW$_{el}$", "hydrogen": r"GW$_{H2}$", "heat": r"GW$_{th}$"}
commodities = {"electricity", "hydrogen", "heat"}
numberOfTimeSteps = 4
hoursPerTimeStep = 1

In [249]:
esM = fn.EnergySystemModel(
    locations=locations,
    commodities=commodities,
    numberOfTimeSteps=4,
    commodityUnitsDict=commodityUnitDict,
    hoursPerTimeStep=1,
    costUnit="1e9 Euro",
    lengthUnit="km",
    verboseLogLevel=0,
)

In [250]:
windOperationRateMax = pd.DataFrame({
    "Region1": [
        1,
        1,
        1,
        1,
    ],
    "Region2": [
        1,
        1,
        1,
        1,
    ],
    "Region3": [
        1,
        1,
        1,
        1,
    ]
})

print(windOperationRateMax)

esM.add(
    fn.Source(
        esM=esM,
        name="Wind",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=windOperationRateMax,
        capacityMax=4e6,
        investPerCapacity=2 * 2190,
        opexPerCapacity=0,
        interestRate=0,
        opexPerOperation=0,
        economicLifetime=1,
    )
)

   Region1  Region2  Region3
0        1        1        1
1        1        1        1
2        1        1        1
3        1        1        1


In [251]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="Electrolyzer",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": -1, "heat": -0.7, "hydrogen": 0.7},
        hasCapacityVariable=True,
        investPerCapacity=0.5,
        opexPerCapacity=0.5 * 0.025,
        interestRate=0.08,
        economicLifetime=10,
    )
)

In [252]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Li-ion batteries",
        commodity="electricity",
        hasCapacityVariable=True,
        chargeEfficiency=0.95,
        cyclicLifetime=10000,
        dischargeEfficiency=0.95,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        doPreciseTsaModeling=False,
        investPerCapacity=0.151,
        opexPerCapacity=0.002,
        interestRate=0.08,
        economicLifetime=22,
    )
)

In [253]:
distances = np.array(
    [
        [0, 1, 1],
        [1, 0, 1],
        [1, 1, 0]
    ]
)


# incidence = np.array(
#     [
#         [0, 1, 1],
#         [1, 0, 1],
#         [1, 1, 0]
#     ]
# )

incidence = np.array(
    [
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]
    ]
)

locs = sorted(locations)   # oder list(locations)

distances = pd.DataFrame(distances, index=locs, columns=locs)
eligibility = pd.DataFrame(incidence, index=locs, columns=locs)

esM.add(
    fn.Transmission(
        esM=esM,
        name="AC cable",
        commodity="electricity",
        losses=0,
        distances=distances,
        hasCapacityVariable=True,
        locationalEligibility=eligibility,
        investPerCapacity=0.1,
        interestRate=0.08,
        economicLifetime=50,
    )
)

In [254]:
distances = np.array(
    [
        [0, 1, 1],
        [1, 0, 1],
        [1, 1, 0]
    ]
)


# incidence = np.array(
#     [
#         [0, 1, 0],
#         [1, 0, 0],
#         [0, 0, 0]
#     ]
# )

incidence = np.array(
    [
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]
    ]
)

locs = sorted(locations)   # oder list(locations)

distances = pd.DataFrame(distances, index=locs, columns=locs)
eligibility = pd.DataFrame(incidence, index=locs, columns=locs)

esM.add(
    fn.Transmission(
        esM=esM,
        name="hydrogen pipeline",
        commodity="hydrogen",
        losses=0,
        distances=distances,
        hasCapacityVariable=True,
        locationalEligibility=eligibility,
        investPerCapacity=0.1,
        interestRate=0.08,
        economicLifetime=50,
    )
)

In [255]:
Demand = pd.DataFrame({
    "Region1": [
        1,
        1,
        1,
        1,
    ],
    "Region2": [
        0,
        0,
        0,
        0,
    ],
    "Region3": [
        0,
        0,
        0,
        0,
    ],
})

esM.add(
    fn.Sink(
        esM=esM,
        name="Hydrogen demand",
        commodity="hydrogen",
        #commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=Demand,
    )
)

# esM.add(
#     fn.Sink(
#         esM=esM,
#         name="CO2 demand",
#         commodity="CO2",
#         #commodity="electricity",
#         hasCapacityVariable=False,
#         operationRateFix=Demand,
#     )
# )

#esM.declareOptimizationProblem()

In [256]:
# import pandas as pd


# def _region_has_positive_demand(sink, region):
#     """
#     Prüft, ob ein Sink in einer Region einen positiven Demand hat.
#     Relevant sind typischerweise operationRateFix oder operationRateMin.
#     """

#     for attr in ["operationRateFix", "operationRateMin"]:
#         value = getattr(sink, attr, None)

#         if value is None:
#             continue

#         # Zeitreihe als DataFrame
#         if isinstance(value, pd.DataFrame):
#             if region in value.columns and (value[region] > 0).any():
#                 return True

#         # Regionaler Series-Demand
#         elif isinstance(value, pd.Series):
#             if region in value.index and value.loc[region] > 0:
#                 return True

#         # Skalarer Demand
#         elif isinstance(value, (int, float)):
#             if value > 0:
#                 return True

#     return False


# def check_sink_commodities_can_be_supplied(esM):
#     """
#     Prüft:
#     Wenn ein Sink in einer Region positive Nachfrage nach Commodity X hat,
#     muss es in derselben Region entweder
#     - eine Source für Commodity X geben oder
#     - eine Conversion geben, die Commodity X erzeugt.
#     """

#     errors = []

#     supplied_commodities_by_region = {
#         loc: set() for loc in esM.locations
#     }

#     # -------------------------
#     # Sources und Conversions sammeln
#     # -------------------------
#     for model in esM.componentModelingDict.values():
#         for comp_name, comp in model.componentsDict.items():

#             comp_type = comp.__class__.__name__

#             # Source stellt ihre commodity bereit
#             if comp_type == "Source":
#                 commodity = comp.commodity

#                 for loc in comp.processedLocationalEligibility.index:
#                     if comp.processedLocationalEligibility.loc[loc] == 1:
#                         supplied_commodities_by_region[loc].add(commodity)

#             # Conversion stellt Commodities mit positivem Faktor bereit
#             elif comp_type == "Conversion":
#                 for commodity, factor in comp.commodityConversionFactors.items():
#                     if factor > 0:
#                         for loc in comp.processedLocationalEligibility.index:
#                             if comp.processedLocationalEligibility.loc[loc] == 1:
#                                 supplied_commodities_by_region[loc].add(commodity)

#             # elif comp_type == "Transmission":
#             #     for commodity, factor in comp.commodityConversionFactors.items():
#             #         if factor > 0:
#             #             for loc in comp.processedLocationalEligibility.index:
#             #                 if comp.processedLocationalEligibility.loc[loc] == 1:
#             #                     supplied_commodities_by_region[loc].add(commodity)

#     # -------------------------
#     # Sinks prüfen
#     # -------------------------
#     for model in esM.componentModelingDict.values():
#         for comp_name, comp in model.componentsDict.items():

#             if comp.__class__.__name__ != "Sink":
#                 continue

#             sink_commodity = comp.commodity

#             for loc in esM.locations:
#                 if not _region_has_positive_demand(comp, loc):
#                     continue

#                 if sink_commodity not in supplied_commodities_by_region[loc]:
#                     errors.append(
#                         f"Sink '{comp_name}' has positive demand for commodity "
#                         f"'{sink_commodity}' in region '{loc}', but this commodity "
#                         f"cannot be supplied there by any Source or Conversion."
#                     )

#     if errors:
#         raise ValueError(
#             "Commodity supply check failed:\n\n" + "\n".join(errors)
#         )

#     print("Commodity supply check passed.")

# check_sink_commodities_can_be_supplied(esM)

In [ ]:
def check_sink_commodities_producible(esM, raise_error=True):
    """
    Prüft, ob alle Sink-Commodities aus Sources und Conversion-Komponenten
    erzeugt werden können.

    Wichtig:
    Eine Conversion darf nur verwendet werden, wenn ALLE Input-Commodities
    dieser Conversion bereits verfügbar sind.
    """

    ssm = esM.componentModelingDict.get("SourceSinkModel")
    conv_model = esM.componentModelingDict.get("ConversionModel")

    source_commodities = set()
    sink_commodities = set()

    # -------------------------
    # Sources und Sinks sammeln
    # -------------------------
    if ssm is not None:
        for comp in ssm.componentsDict.values():
            if getattr(comp, "sign", None) == 1:
                source_commodities.add(comp.commodity)

            elif getattr(comp, "sign", None) == -1:
                sink_commodities.add(comp.commodity)

    producible_commodities = set(source_commodities)

    # -------------------------
    # Conversions iterativ anwenden
    # -------------------------
    changed = True

    while changed:
        changed = False

        if conv_model is None:
            break

        for comp in conv_model.componentsDict.values():
            factors = comp.commodityConversionFactors

            input_commodities = {
                com for com, factor in factors.items()
                if factor < 0
            }

            output_commodities = {
                com for com, factor in factors.items()
                if factor > 0
            }

            # Conversion ist nur nutzbar, wenn ALLE Inputs verfügbar sind
            if input_commodities.issubset(producible_commodities):
                before = len(producible_commodities)
                producible_commodities.update(output_commodities)

                if len(producible_commodities) > before:
                    changed = True

    missing_sink_commodities = sink_commodities - producible_commodities

    if missing_sink_commodities:
        message = (
            "The following sink commodities cannot be produced from the "
            "available source commodities and conversion components:\n"
            + "\n".join(f"- {com}" for com in sorted(missing_sink_commodities))
            + "\n\nAvailable source commodities:\n"
            + "\n".join(f"- {com}" for com in sorted(source_commodities))
            + "\n\nProducible commodities:\n"
            + "\n".join(f"- {com}" for com in sorted(producible_commodities))
        )

        if raise_error:
            raise ValueError(message)

        return missing_sink_commodities

    return set()

set()